In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/appledisease/Test/Blotch_Apple/149.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/143.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/131.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/130.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/120.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/124.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/123.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/137.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/127.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/140.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/145.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/135.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/121.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/126.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/142.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/148.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/138.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/146.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/1

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import ConvNeXtBase
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import os

2025-11-27 23:41:05.457463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764286865.636792      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764286865.686924      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
train_dir = "/kaggle/input/appledisease/Train"
test_dir = "/kaggle/input/appledisease/Test"


In [4]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [5]:
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
train_gen = train_datagen.flow_from_directory(
    train_dir, 
    target_size=img_size, 
    batch_size=batch_size,
    class_mode='categorical', 
    subset='training', 
    shuffle=True)

Found 307 images belonging to 4 classes.


In [7]:
val_gen = val_datagen.flow_from_directory(
    train_dir, 
    target_size=img_size, 
    batch_size=batch_size,
    class_mode='categorical', 
    subset='validation', 
    shuffle=False)

Found 75 images belonging to 4 classes.


In [8]:
test_gen = test_datagen.flow_from_directory(
    test_dir, 
    target_size=img_size, 
    batch_size=batch_size,
    class_mode='categorical', 
    shuffle=False)

Found 120 images belonging to 4 classes.


In [9]:
for subdir, _, files in os.walk(train_dir):
    for file in files:
        try:
            img_path = os.path.join(subdir, file)
            img = Image.open(img_path)
            img.verify()
        except Exception as e:
            print(f"Corrupted or unreadable file: {img_path} — {e}")

In [10]:
labels = train_gen.classes
class_weight = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight = dict(enumerate(class_weight))

In [11]:
base_model = ConvNeXtBase(
    include_top=False, weights="imagenet", input_shape=(224, 224, 3))
base_model.trainable = False

I0000 00:00:1764286904.709685      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


350926856/350926856 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [12]:
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(train_gen.num_classes, activation='softmax'))

In [13]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])

In [14]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, verbose=1)

In [15]:
history1=model.fit(train_gen, 
          validation_data=val_gen, 
          epochs=10,
          class_weight=class_weight, 
          callbacks=[early_stop, lr_reduce]
         )

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1764286940.277503     117 service.cc:148] XLA service 0x7c5064016e50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764286940.278177     117 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1764286943.114551     117 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1764286948.520974     117 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10/10 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.2139 - loss: 2.1570 - val_accuracy: 0.1733 - val_loss: 1.4976 - learning_rate: 1.0000e-04
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 577ms/step - accuracy: 0.2079 - loss: 2.0356 - val_accuracy: 0.4400 - val_loss: 1.3128 - learning_rate: 1.0000e-04
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 556ms/step - accuracy: 0.2397 - loss: 1.8076 - val_accuracy: 0.4933 - val_loss: 1.2709 - learning_rate: 1.0000e-04
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 548ms/step - accuracy: 0.2910 - loss: 1.7373 - val_accuracy: 0.5200 - val_loss: 1.2432 - learning_rate: 1.0000e-04
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 571ms/step - accuracy: 0.3324 - loss: 1.5839 - val_accuracy: 0.5333 - val_loss: 1.2141 - learning_rate: 1.0000e-04
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 599ms/step - accuracy: 0.2751 - loss: 1.6131 - val_accuracy: 0.6000 - val_loss: 1.1897 - learning_rate: 1.0000e-04
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 556ms/step - accuracy: 0.3421 - loss

In [16]:
print("\nPhase 2: Fine-tuning ConvNeXt")
base_model.trainable = True


Phase 2: Fine-tuning ConvNeXt


In [17]:
for layer in base_model.layers[:200]:
    layer.trainable = False

In [18]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
early_stop2 = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
lr_reduce2 = ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, verbose=1)

In [20]:
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=12,
    class_weight=class_weight,
    callbacks=[early_stop2, lr_reduce2]
)

Epoch 1/12


E0000 00:00:1764287089.893805     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764287090.079987     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764287090.354388     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764287090.564649     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 3/10 ━━━━━━━━━━━━━━━━━━━━ 2s 417ms/step - accuracy: 0.4080 - loss: 1.3861

E0000 00:00:1764287105.927433     116 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764287106.115674     116 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764287106.393648     116 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764287106.604174     116 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


10/10 ━━━━━━━━━━━━━━━━━━━━ 78s 3s/step - accuracy: 0.4245 - loss: 1.3918 - val_accuracy: 0.5467 - val_loss: 1.0729 - learning_rate: 1.0000e-05
Epoch 2/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 559ms/step - accuracy: 0.4153 - loss: 1.3051 - val_accuracy: 0.5467 - val_loss: 1.0448 - learning_rate: 1.0000e-05
Epoch 3/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 613ms/step - accuracy: 0.4104 - loss: 1.3431 - val_accuracy: 0.5867 - val_loss: 1.0160 - learning_rate: 1.0000e-05
Epoch 4/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 626ms/step - accuracy: 0.4137 - loss: 1.3726 - val_accuracy: 0.5600 - val_loss: 1.0073 - learning_rate: 1.0000e-05
Epoch 5/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 531ms/step - accuracy: 0.4512 - loss: 1.2461 - val_accuracy: 0.6133 - val_loss: 0.9600 - learning_rate: 1.0000e-05
Epoch 6/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 589ms/step - accuracy: 0.4918 - loss: 1.1008 - val_accuracy: 0.6800 - val_loss: 0.9241 - learning_rate: 1.0000e-05
Epoch 7/12
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 621ms/step - accuracy: 0.4337 - loss

In [21]:
best_train_acc = max(history2.history["accuracy"])
best_val_acc = max(history2.history["val_accuracy"])

In [22]:
print(f"Best Training Accuracy: {best_train_acc:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

Best Training Accuracy: 0.5505
Best Validation Accuracy: 0.6800


In [23]:
test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.5750
